# Grasp adaptation — compliance-matched stiffness control

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
from pathlib import Path

sys.path.insert(0, os.path.join('..'))
plt.style.use(os.path.join('..', 'plot_config.mplstyle'))

OUTPUT_DIR = os.path.join('outputs', 'grasp_adaptation')
OBJECTS    = ['object_1', 'object_2']
COLORS     = {'object_1': '#0072B2', 'object_2': '#D55E00'}

def load_latest(obj):
    folder = Path(OUTPUT_DIR)
    if not folder.exists():
        return None
    files = sorted(folder.glob(f'grasp_{obj}_*.csv'))
    return pd.read_csv(files[-1]) if files else None

data = {obj: load_latest(obj) for obj in OBJECTS}

## Sensing phase — fingertip displacement per object

In [ ]:
fig, ax = plt.subplots()

FINGERTIPS = ['thumb', 'index', 'middle', 'ring', 'pinky']
x = np.arange(len(FINGERTIPS))
width = 0.35

for i, (obj, color) in enumerate(COLORS.items()):
    df = data[obj]
    if df is None:
        continue
    sense = df[df['phase'] == 'sense']
    means = [sense[f'disp_{f}_mag_m'].mean() * 1e3 for f in FINGERTIPS]
    ax.bar(x + i * width, means, width, label=obj.replace('_', ' ').title(),
           color=color)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(FINGERTIPS)
ax.set_xlabel('Finger')
ax.set_ylabel('Mean tip displacement [mm]')
ax.legend()
fig.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
fig.savefig(os.path.join(OUTPUT_DIR, 'sensing_displacement.pdf'), bbox_inches='tight')
plt.show()

## Adapted stiffness — sensing displacement vs k_applied

In [ ]:
fig, ax = plt.subplots()

# Overlay the adaptation curve k = K_SCALE * delta (clipped to [K_MIN, K_MAX]).
_ns = {}
with open('grasp_adaptation.py') as _f:
    for _line in _f:
        if _line.startswith(('K_SCALE', 'K_MIN', 'K_MAX')):
            exec(_line, _ns)
K_SCALE = float(_ns['K_SCALE'])
K_MIN   = float(_ns['K_MIN'])
K_MAX   = float(_ns['K_MAX'])

delta_range = np.linspace(0.0, 0.05, 300)
k_curve     = np.clip(K_SCALE * delta_range, K_MIN, K_MAX)
ax.plot(delta_range * 1e3, k_curve, color='0.5', lw=1, linestyle='--',
        label='adaptation curve')

for obj, color in COLORS.items():
    df = data[obj]
    if df is None:
        continue
    sense = df[df['phase'] == 'sense']
    delta = float(sense['delta_mean_m'].mean()) * 1e3
    k_app = float(sense['k_applied_Npm'].iloc[-1])
    ax.scatter([delta], [k_app], color=color, zorder=5,
               label=obj.replace('_', ' ').title())
    ax.annotate(f'{k_app:.0f} N/m', (delta, k_app),
                textcoords='offset points', xytext=(6, 4))

ax.set_xlabel('δ_mean [mm]')
ax.set_ylabel('k_applied [N/m]')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'adaptation_curve.pdf'), bbox_inches='tight')
plt.show()